In [ ]:
import requests
import pandas as pd
import time

# ✅ Replace this with your actual RapidAPI Key
headers = {
    "X-RapidAPI-Key": "e51124b757msh36c50ee61bc19c9p107e06jsn8f4d7a294f3a",
    "X-RapidAPI-Host": "imdb236.p.rapidapi.com"
}

# ✅ Step 1: Fetch all IMDb top-rated movie IDs
def fetch_imdb_ids():
    url = "https://api/imdb/tt0816692"
    querystring = {"homeCountry":"US","purchaseCountry":"US","currentCountry":"US"}
    response = requests.get(url, headers=headers, params=querystring)
    if response.status_code == 200:
        ids = response.json()
        return [i.split('/')[2] for i in ids]  # Extract ttXXXXXX
    else:
        print("Failed to get top movie list")
        return []

# ✅ Step 2: Fetch all details using all endpoints
def get_movie_data(imdb_id):
    movie = {"imdb_id": imdb_id}

    try:
        # 1️⃣ Overview Details
        res = requests.get("https://imdb8.p.rapidapi.com/title/get-overview-details",
                           headers=headers, params={"tconst": imdb_id, "currentCountry": "US"})
        if res.status_code == 200:
            data = res.json()
            movie["title"] = data.get("title", {}).get("title")
            movie["year"] = data.get("title", {}).get("year")
            movie["rating"] = data.get("ratings", {}).get("rating")
            movie["metascore"] = data.get("metacritic", {}).get("metascore")
            movie["poster"] = data.get("title", {}).get("image", {}).get("url")

        # 2️⃣ Top Crew (Directors & Writers)
        res = requests.get("https://imdb8.p.rapidapi.com/title/get-top-crew",
                           headers=headers, params={"tconst": imdb_id})
        if res.status_code == 200:
            data = res.json()
            movie["directors"] = ', '.join(data.get("directors", []))
            movie["writers"] = ', '.join(data.get("writers", []))

        # 3️⃣ Top Cast
        res = requests.get("https://imdb8.p.rapidapi.com/title/get-cast",
                           headers=headers, params={"tconst": imdb_id})
        if res.status_code == 200:
            data = res.json()
            cast_names = [i["name"]["name"] for i in data[:5]]
            movie["top_cast"] = ', '.join(cast_names)

        # 4️⃣ Genres (optional)
        res = requests.get("https://imdb8.p.rapidapi.com/title/get-genres",
                           headers=headers, params={"tconst": imdb_id})
        if res.status_code == 200:
            genres = res.json()
            movie["genres"] = ', '.join(genres)

        # 5️⃣ TMDB Movie Info (optional)
        res = requests.get("https://imdb8.p.rapidapi.com/title/get-tmdb-movie-details",
                           headers=headers, params={"tconst": imdb_id})
        if res.status_code == 200:
            data = res.json()
            movie["tmdb_rating"] = data.get("vote_average")
            movie["tmdb_vote_count"] = data.get("vote_count")

    except Exception as e:
        print(f"Error fetching {imdb_id}: {e}")

    return movie

# ✅ Step 3: Run the collection loop
imdb_ids = fetch_imdb_ids()
print(f"🎬 Total movies found: {len(imdb_ids)}")

all_movies = []
for i, imdb_id in enumerate(imdb_ids):
    print(f"🔄 ({i+1}/{len(imdb_ids)}) Processing: {imdb_id}")
    movie = get_movie_data(imdb_id)
    all_movies.append(movie)
    time.sleep(1)  # To avoid hitting rate limits

# ✅ Step 4: Save to CSV
df = pd.DataFrame(all_movies)
df.to_csv("imdb_full_data.csv", index=False)
print("✅ All done! Saved as imdb_full_data.csv")
